In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 23.8 MB/s eta 0:00:00


In [ ]:
# ============================================================
#  CodeT5-Small Fine-Tuning với LoRA trên Spider (Text-to-SQL)
#  Tối ưu cho: Ít Epoch – Độ chính xác cao nhất – Tiết kiệm VRAM
#  Model: Salesforce/codet5-small (~60M params, pretrain trên code)
#  CHẠY TRÊN GOOGLE COLAB - ĐÃ TẮT EARLY STOPPING
# ============================================================

!pip install -q "transformers>=4.41.0,<5.0.0" datasets sentencepiece accelerate tqdm peft
!pip install -q kaggle

import os, json, re, shutil, random, math, time, zipfile, urllib.request
import numpy as np
import nltk
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
    DataCollatorForSeq2Seq,
)
from tqdm.auto import tqdm

# Import các module của PEFT để cấu hình LoRA
from peft import LoraConfig, get_peft_model, TaskType

# ── 1. SETUP ──────────────────────────────────────────────────
os.environ['KAGGLE_USERNAME'] = "phankhaclap"
os.environ['KAGGLE_KEY']      = "0ba946628cb1f5acb76ecd357f590e95"

# Chuyển sang đường dẫn kết quả cho biến thể LoRA
FINAL_SAVE_PATH = "/content/drive/MyDrive/CodeT5-small_LoRA"
CHECKPOINT_DIR  = os.path.join(FINAL_SAVE_PATH, "checkpoints")
RESUME_DIR      = os.path.join(FINAL_SAVE_PATH, "resume")

for d in [FINAL_SAVE_PATH, CHECKPOINT_DIR, RESUME_DIR]:
    os.makedirs(d, exist_ok=True)

RESUME_STATE_FILE = os.path.join(RESUME_DIR, "training_state.json")
RESUME_MODEL_DIR  = os.path.join(RESUME_DIR, "model")
RESUME_OPT_FILE   = os.path.join(RESUME_DIR, "optimizer.pt")

# ── 2. TẢI DỮ LIỆU ────────────────────────────────────────────
print(">>> [1/7] Tải dữ liệu Spider...")
if os.path.exists('spider_data'):
    shutil.rmtree('spider_data')

!kaggle datasets download -d jeromeblanchet/yale-universitys-spider-10-nlp-dataset
zip_path = "yale-universitys-spider-10-nlp-dataset.zip"

if os.path.exists(zip_path):
    print("Đang giải nén dữ liệu...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("temp_spider")
    if os.path.exists("temp_spider/spider"):
        shutil.move("temp_spider/spider", "spider_data")
    else:
        shutil.rename("temp_spider", "spider_data")
    if os.path.exists('temp_spider'):
        shutil.rmtree('temp_spider', ignore_errors=True)
    os.remove(zip_path)
else:
    print("❌ KHÔNG TÌM THẤY FILE ZIP!")

print("Đang tải công cụ chấm điểm chính thức của Spider...")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py",
    "evaluation.py"
)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py",
    "process_sql.py"
)

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
print("Xong bước tải dữ liệu.")

# ── 3. CẤU HÌNH ───────────────────────────────────────────────
CFG = dict(
    model_name      = "Salesforce/codet5-small",
    max_input_len   = 512,
    max_target_len  = 256,
    batch_size      = 8,
    grad_accum      = 4,
    num_epochs      = 5,
    lr              = 1e-3, # LoRA thường hội tụ tốt hơn ở mức LR lớn hơn FFT (ví dụ: 5e-4 đến 1e-3)
    warmup_ratio    = 0.06,
    weight_decay    = 0.01,
    fp16            = True,
    beam_size       = 6,
    seed            = 42,
    label_smoothing = 0.1,
    question_budget = 96,
    length_penalty  = 0.8,
    num_workers     = 2,
    # Cấu hình siêu tham số cho LoRA
    lora_r          = 64,
    lora_alpha      = 128,
    lora_dropout    = 0.05
)

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_FP16 = CFG['fp16'] and (DEVICE.type == 'cuda')
print(f"Thiết bị: {DEVICE}  |  Dùng FP16: {USE_FP16}")

# ── 4. TIỀN XỬ LÝ ────────────────────────────────────────────
print(">>> [2/7] Tiền xử lý dữ liệu...")

def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

TYPE_SHORT = {"text": "T", "number": "N", "time": "D", "boolean": "B", "others": "O"}

def build_schema_map(tables_path):
    tables = load_json(tables_path)
    schema_map = {}
    for db in tables:
        db_id     = db['db_id']
        col_types = db.get('column_types', [])
        parts     = []
        for t_idx, t_name in enumerate(db['table_names_original']):
            cols = []
            for c_idx, (t_i, c_name) in enumerate(db['column_names_original']):
                if t_i == t_idx:
                    ct = TYPE_SHORT.get(
                        col_types[c_idx] if c_idx < len(col_types) else "others", "O"
                    )
                    cols.append(f"{c_name}:{ct}")
            parts.append(f"{t_name}({','.join(cols)})")
        schema_map[db_id] = " | ".join(parts)
    return schema_map

def normalize_sql(sql: str) -> str:
    sql = sql.lower().strip()
    sql = re.sub(r'\s+', ' ', sql)
    sql = re.sub(r'\s*([,\(\)])\s*', r' \1 ', sql)
    return re.sub(r'\s+', ' ', sql).strip()

PREFIX     = "Translate English to SQL: "
SEP        = " | schema: "
PREFIX_TOK = 8

def build_input_smart(question: str, schema: str, tokenizer=None, max_len=512) -> str:
    if tokenizer is None:
        return f"{PREFIX}{question.strip()}{SEP}{schema}"
    q_budget = CFG['question_budget']
    s_budget = max_len - PREFIX_TOK - q_budget
    q_ids = tokenizer.encode(
        question.strip(), add_special_tokens=False,
        max_length=q_budget, truncation=True
    )
    s_ids = tokenizer.encode(
        schema, add_special_tokens=False,
        max_length=s_budget, truncation=True
    )
    q_text = tokenizer.decode(q_ids, skip_special_tokens=True)
    s_text = tokenizer.decode(s_ids, skip_special_tokens=True)
    return f"{PREFIX}{q_text}{SEP}{s_text}"

def load_spider_split(data_path, schema_map, tokenizer=None):
    samples = []
    for item in load_json(data_path):
        db_id  = item['db_id']
        schema = schema_map.get(db_id, "")
        inp    = build_input_smart(item['question'], schema, tokenizer, CFG['max_input_len'])
        samples.append({
            "input":    inp,
            "target":   item['query'],
            "db_id":    db_id,
            "question": item['question']
        })
    return samples

schema_map = build_schema_map("spider_data/tables.json")
tokenizer  = AutoTokenizer.from_pretrained(CFG['model_name'])

train_data = load_spider_split("spider_data/train_spider.json", schema_map, tokenizer)
dev_data   = load_spider_split("spider_data/dev.json", schema_map, tokenizer)

others_path = "spider_data/train_others.json"
if os.path.exists(others_path):
    train_data += load_spider_split(others_path, schema_map, tokenizer)

random.shuffle(train_data)
print(f"Train: {len(train_data)} mẫu  |  Dev: {len(dev_data)} mẫu")

# ── 5. DATASET & DATALOADER ───────────────────────────────────
print(">>> [3/7] Đóng gói Dữ liệu (Tokenization)...")

class SpiderDataset(Dataset):
    def __init__(self, samples, tokenizer, max_in, max_out):
        self.samples   = samples
        self.tokenizer = tokenizer
        self.max_in    = max_in
        self.max_out   = max_out

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        enc = self.tokenizer(
            s['input'], max_length=self.max_in, truncation=True, padding=False
        )
        tgt = self.tokenizer(
            text_target=s['target'], max_length=self.max_out, truncation=True, padding=False
        )
        labels = [
            l if l != self.tokenizer.pad_token_id else -100
            for l in tgt['input_ids']
        ]
        return {
            'input_ids':      enc['input_ids'],
            'attention_mask': enc['attention_mask'],
            'labels':         labels,
            'target_sql':     s['target'],
            'db_id':          s['db_id'],
            'question':       s['question']
        }

train_ds = SpiderDataset(train_data, tokenizer, CFG['max_input_len'], CFG['max_target_len'])
dev_ds   = SpiderDataset(dev_data,   tokenizer, CFG['max_input_len'], CFG['max_target_len'])

collator = DataCollatorForSeq2Seq(
    tokenizer, model=None, label_pad_token_id=-100, pad_to_multiple_of=8
)

def collate_fn(batch):
    meta     = {k: [b.pop(k) for b in batch] for k in ('target_sql', 'db_id', 'question')}
    collated = collator(batch)
    collated.update(meta)
    return collated

train_loader = DataLoader(
    train_ds,
    batch_size=CFG['batch_size'],
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=CFG['num_workers'],
    pin_memory=True
)
dev_loader = DataLoader(
    dev_ds,
    batch_size=CFG['batch_size'] * 2,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=CFG['num_workers'],
    pin_memory=True
)

# ── 6. LOSS FUNCTION ──────────────────────────────────────────
loss_fn = nn.CrossEntropyLoss(
    label_smoothing=CFG['label_smoothing'], ignore_index=-100
)

def compute_loss(logits, labels):
    B, T, V = logits.shape
    return loss_fn(logits.reshape(B * T, V), labels.reshape(B * T))

# ── 7. MODEL & OPTIMIZER VỚI TÍCH HỢP LORA ─────────────────────
print(f">>> [4/7] Khởi tạo CodeT5Small + Cấu hình LoRA Adapters...")

total_steps  = math.ceil(len(train_loader) / CFG['grad_accum']) * CFG['num_epochs']
warmup_steps = int(total_steps * CFG['warmup_ratio'])
no_decay     = ["bias", "LayerNorm.weight"]

def apply_lora_config(base_model):
    """Bọc mô hình nền bằng cấu hình cấu trúc LoRA"""
    lora_config = LoraConfig(
        r=CFG['lora_r'],
        lora_alpha=CFG['lora_alpha'],
        target_modules=["q", "k", "v", "o", "wi_0", "wi_1", "wo"],
        lora_dropout=CFG['lora_dropout'],
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM
    )
    model_with_lora = get_peft_model(base_model, lora_config)
    return model_with_lora

def build_optimizer_scheduler(model):
    # Chỉ tối ưu hóa các tham số LoRA có thuộc tính requires_grad=True
    param_groups = [
        {
            "params": [
                p for n, p in model.named_parameters()
                if p.requires_grad and not any(nd in n for nd in no_decay)
            ],
            "weight_decay": CFG['weight_decay']
        },
        {
            "params": [
                p for n, p in model.named_parameters()
                if p.requires_grad and any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.0
        },
    ]
    opt   = AdamW(param_groups, lr=CFG['lr'], eps=1e-8, betas=(0.9, 0.98))
    sched = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)
    return opt, sched

START_EPOCH, history, global_step = 1, [], 0

if os.path.exists(RESUME_STATE_FILE) and os.path.exists(RESUME_MODEL_DIR):
    print("🔄 Resume checkpoint LoRA phát hiện...")
    state = load_json(RESUME_STATE_FILE)

    # Tải lại mô hình nền ban đầu
    raw_base_model = AutoModelForSeq2SeqLM.from_pretrained(CFG['model_name']).to(DEVICE)
    # Khôi phục lại cấu trúc LoRA tương ứng
    model = apply_lora_config(raw_base_model)

    # Nạp đè trọng số Adapter từ thư mục lưu trữ thủ công
    from peft import set_peft_model_state_dict
    adapters_weights = torch.load(os.path.join(RESUME_MODEL_DIR, "adapter_model.bin"), map_location=DEVICE)
    set_peft_model_state_dict(model, adapters_weights)

    optimizer, scheduler = build_optimizer_scheduler(model)
    if os.path.exists(RESUME_OPT_FILE):
        opt_checkpoint = torch.load(RESUME_OPT_FILE, map_location=DEVICE)
        optimizer.load_state_dict(opt_checkpoint['optimizer'])
        scheduler.load_state_dict(opt_checkpoint['scheduler'])
    scaler = GradScaler('cuda', enabled=USE_FP16)
    if 'opt_checkpoint' in locals() and 'scaler' in opt_checkpoint:
        scaler.load_state_dict(opt_checkpoint['scaler'])

    global_step  = state['global_step']
    START_EPOCH  = state['epoch'] + 1
    history      = state.get('history', [])
else:
    raw_base_model = AutoModelForSeq2SeqLM.from_pretrained(
        CFG['model_name'],
        use_safetensors=True
    ).to(DEVICE)
    model = apply_lora_config(raw_base_model)
    optimizer, scheduler = build_optimizer_scheduler(model)
    scaler = GradScaler('cuda', enabled=USE_FP16)

# Hiển thị số lượng tham số cần huấn luyện thực tế
model.print_trainable_parameters()

def save_resume_checkpoint(model, optimizer, scheduler, scaler, epoch, global_step, history):
    # PEFT tự động chỉ lưu cấu hình và file tệp tin trọng số adapter rất nhẹ
    model.save_pretrained(RESUME_MODEL_DIR)
    tokenizer.save_pretrained(RESUME_MODEL_DIR)
    torch.save({
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler':    scaler.state_dict()
    }, RESUME_OPT_FILE)
    with open(RESUME_STATE_FILE, 'w', encoding='utf-8') as f:
        json.dump({
            'epoch':        epoch,
            'global_step':  global_step,
            'history':      history,
            'config':       CFG
        }, f, indent=2)


# ── 8. PERFORMANCE METRICS ───────────────────────────────────
def measure_model_size_mb(model_path: str) -> float:
    total_bytes = 0
    if os.path.exists(model_path):
        for root, dirs, files in os.walk(model_path):
            for fname in files:
                if fname.endswith(('.bin', '.safetensors', '.pt', '.json')):
                    total_bytes += os.path.getsize(os.path.join(root, fname))
    return total_bytes / (1024 ** 2)


def measure_latency_throughput(model, loader, n_warmup_batches: int = 3):
    model.eval()
    latencies   = []
    total_samps = 0

    with torch.no_grad():
        for i, batch in enumerate(tqdm(loader, desc="⏱ Đo Latency/Throughput", leave=False)):
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            bsz            = input_ids.size(0)

            if i < n_warmup_batches:
                model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=CFG['max_target_len'],
                    num_beams=CFG['beam_size'],
                    early_stopping=True,
                    length_penalty=CFG['length_penalty'],
                )
                continue

            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            t_start = time.perf_counter()

            model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=CFG['max_target_len'],
                num_beams=CFG['beam_size'],
                early_stopping=True,
                length_penalty=CFG['length_penalty'],
            )

            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            t_end = time.perf_counter()

            latencies.append(t_end - t_start)
            total_samps += bsz

    model.train()
    if not latencies:
        return 0.0, 0.0

    total_time_s      = sum(latencies)
    latency_ms_sample = (total_time_s / total_samps) * 1000
    throughput_sps    = total_samps / total_time_s
    return round(latency_ms_sample, 2), round(throughput_sps, 2)


def measure_peak_vram_mb(model, loader, n_batches: int = 5) -> float:
    if DEVICE.type != 'cuda':
        return 0.0

    model.eval()
    torch.cuda.reset_peak_memory_stats(DEVICE)

    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches:
                break
            model.generate(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE),
                max_new_tokens=CFG['max_target_len'],
                num_beams=CFG['beam_size'],
                early_stopping=True,
                length_penalty=CFG['length_penalty'],
            )

    peak_bytes = torch.cuda.max_memory_allocated(DEVICE)
    model.train()
    return round(peak_bytes / (1024 ** 2), 2)


def print_performance_report(metrics: dict):
    print("\n" + "=" * 55)
    print("📊 PERFORMANCE METRICS REPORT (LoRA VARIANT)")
    print("=" * 55)
    print(f"{'Chỉ số':<28} {'Giá trị':>15}")
    print("-" * 55)
    print(f"{'Adapter Size (MB)':<28} {metrics['model_size_mb']:>14.2f} MB")
    print(f"{'Latency (ms/sample)':<28} {metrics['latency_ms']:>14.2f} ms")
    print(f"{'Throughput (samples/s)':<28} {metrics['throughput_sps']:>14.2f} s/s")
    vram_str = (
        f"{metrics['peak_vram_mb']:.2f} MB"
        if metrics['peak_vram_mb'] > 0 else "N/A (CPU)"
    )
    print(f"{'Peak VRAM (MB)':<28} {vram_str:>15}")
    print("=" * 55 + "\n")

# ── 9. TRAINING LOOP ──────────────────────────────────────────
print(f"\n>>> [5/7] Bắt đầu huấn luyện với Adapter LoRA ({CFG['num_epochs']} epochs, LR={CFG['lr']:.0e})...")
print(f"Tổng optimizer steps: {total_steps} | Warmup: {warmup_steps} steps")

for epoch in range(START_EPOCH, CFG['num_epochs'] + 1):
    model.train()
    epoch_loss, t0 = 0.0, time.time()
    optimizer.zero_grad()
    train_pbar = tqdm(
        train_loader, desc=f"Epoch {epoch:02d}/{CFG['num_epochs']}", leave=False
    )

    for step, batch in enumerate(train_pbar):
        with autocast('cuda', enabled=USE_FP16):
            outputs = model(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE),
                labels=batch['labels'].to(DEVICE),
            )
            loss = compute_loss(
                outputs.logits, batch['labels'].to(DEVICE)
            ) / CFG['grad_accum']

        scaler.scale(loss).backward()

        if (step + 1) % CFG['grad_accum'] == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

        epoch_loss += loss.item() * CFG['grad_accum']
        train_pbar.set_postfix({'loss': f"{(epoch_loss / (step + 1)):.4f}"})

    avg_loss = epoch_loss / len(train_loader)
    cur_lr  = scheduler.get_last_lr()[0]
    elapsed = time.time() - t0

    history.append({
        "epoch": epoch,
        "loss":  round(avg_loss, 4),
        "lr":    round(cur_lr, 8)
    })

    print(
        f"📊 Epoch {epoch:3d}/{CFG['num_epochs']}: "
        f"Loss={avg_loss:.4f} | LR={cur_lr:.2e} | {elapsed:.0f}s"
    )

    if math.isnan(avg_loss):
        print("❌ Loss = NaN, dừng huấn luyện.")
        break

    # Lưu checkpoint định kỳ của mô hình LoRA
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_ep{epoch}")
    model.save_pretrained(ckpt_path)
    tokenizer.save_pretrained(ckpt_path)

    # Dọn dẹp lưu trữ các checkpoint trung gian lỗi thời
    for d in os.listdir(CHECKPOINT_DIR):
        full = os.path.join(CHECKPOINT_DIR, d)
        if full != ckpt_path and os.path.isdir(full):
            shutil.rmtree(full, ignore_errors=True)

    print(f"✅ Đã lưu checkpoint LoRA tại epoch {epoch}")

    save_resume_checkpoint(
        model, optimizer, scheduler, scaler, epoch,
        global_step, history
    )

# ── 10. LƯU MODEL CUỐI ───────────────────────────────────────
print("\n>>> [6/7] Lưu kết quả cấu hình Adapter LoRA hoàn chỉnh...")
ckpt_dirs = sorted(
    [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("checkpoint_")],
    key=lambda x: int(x.split("ep")[-1])
)
if ckpt_dirs:
    best_dir = os.path.join(CHECKPOINT_DIR, ckpt_dirs[-1])

    # Nạp lại chính xác các tệp weights từ bản checkpoint lưu gần nhất
    from peft import PeftModel
    base_model_reload = AutoModelForSeq2SeqLM.from_pretrained(CFG['model_name']).to(DEVICE)
    final_peft_model = PeftModel.from_pretrained(base_model_reload, best_dir)
    final_peft_model.save_pretrained(FINAL_SAVE_PATH)
    tokenizer.save_pretrained(FINAL_SAVE_PATH)

with open(os.path.join(FINAL_SAVE_PATH, "training_history.json"), "w") as f:
    json.dump({
        "config":     CFG,
        "history":    history
    }, f, indent=2)

# Giải phóng bộ nhớ đệm phụ
for path in [RESUME_STATE_FILE, RESUME_OPT_FILE]:
    if os.path.exists(path):
        os.remove(path)
shutil.rmtree(RESUME_MODEL_DIR, ignore_errors=True)
print("🗑️ Dọn dẹp checkpoint phụ thành công.")

# ── 11. INFERENCE + SPIDER OFFICIAL EVAL + PERFORMANCE METRICS ─
print("\n>>> [7/7] Inference & Spider Official Evaluation & Performance Metrics...")

# Load mô hình hoàn chỉnh (Base + LoRA) từ FINAL_SAVE_PATH để đảm bảo tính độc lập
from peft import PeftModel
base_model_eval = AutoModelForSeq2SeqLM.from_pretrained(CFG['model_name']).to(DEVICE)
infer_model = PeftModel.from_pretrained(base_model_eval, FINAL_SAVE_PATH).to(DEVICE)
infer_model.eval()

# ── 11a. Batch Inference → pred.txt / gold.txt ────────────────
predictions, gold_lines = [], []

for batch in tqdm(dev_loader, desc="Inference", leave=False):
    with torch.no_grad():
        generated = infer_model.generate(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
            max_new_tokens=CFG['max_target_len'],
            num_beams=CFG['beam_size'],
            early_stopping=True,
            length_penalty=CFG['length_penalty'],
        )
    preds = tokenizer.batch_decode(generated, skip_special_tokens=True)
    for pred, gold_sql, db_id in zip(preds, batch['target_sql'], batch['db_id']):
        predictions.append(pred + "\n")
        gold_lines.append(f"{gold_sql}\t{db_id}\n")

with open('pred.txt', 'w', encoding='utf-8') as f:
    f.writelines(predictions)
with open('gold.txt', 'w', encoding='utf-8') as f:
    f.writelines(gold_lines)

# ── 11b. Spider Official Evaluation ───────────────────────────
with open("evaluation.py", "r", encoding="utf-8") as f:
    eval_content = f.read()
eval_content = eval_content.replace(
    'conn = sqlite3.connect(db)',
    'conn = sqlite3.connect(db)\n    conn.text_factory = lambda b: b.decode(errors="ignore")'
)
with open("evaluation.py", "w", encoding="utf-8") as f:
    f.write(eval_content)

print("\n>>> Spider Official Results (Trên Dev set):")
os.system(
    "python evaluation.py "
    "--gold gold.txt --pred pred.txt "
    "--db spider_data/database "
    "--table spider_data/tables.json "
    "--etype all"
)

# ── 11c. Performance Metrics ──────────────────────────────────
print("\n>>> Đo Performance Metrics...")

# Tính toán kích thước của riêng tệp adapter lưu trên đĩa (rất nhẹ, tầm vài chục MB thay vì vài trăm MB)
model_size_mb = measure_model_size_mb(FINAL_SAVE_PATH)

latency_ms, throughput_sps = measure_latency_throughput(
    infer_model, dev_loader, n_warmup_batches=3
)

peak_vram_mb = measure_peak_vram_mb(infer_model, dev_loader, n_batches=5)

perf_metrics = {
    "model_size_mb":  round(model_size_mb, 2),
    "latency_ms":     latency_ms,
    "throughput_sps": throughput_sps,
    "peak_vram_mb":   peak_vram_mb,
}

print_performance_report(perf_metrics)

history_path = os.path.join(FINAL_SAVE_PATH, "training_history.json")
with open(history_path, "r", encoding="utf-8") as f:
    saved_history = json.load(f)

saved_history["performance_metrics"] = perf_metrics

with open(history_path, "w", encoding="utf-8") as f:
    json.dump(saved_history, f, indent=2)

# ── 11d. Lưu kết quả inference ────────────────────────────────
import shutil as _shutil
_shutil.copy('pred.txt', os.path.join(FINAL_SAVE_PATH, 'pred_dev.txt'))
_shutil.copy('gold.txt', os.path.join(FINAL_SAVE_PATH, 'gold_dev.txt'))

print(f"\n✅ Kết quả lưu thành công tại : {FINAL_SAVE_PATH}")
print(f"🏆 Epoch hoàn thành             : {CFG['num_epochs']}")
print(f"💾 Adapter Weight Size          : {perf_metrics['model_size_mb']:.2f} MB")
print(f"⚡ Latency                      : {perf_metrics['latency_ms']:.2f} ms/sample")
print(f"🚀 Throughput                   : {perf_metrics['throughput_sps']:.2f} samples/s")
vram_display = (
    f"{perf_metrics['peak_vram_mb']:.2f} MB"
    if perf_metrics['peak_vram_mb'] > 0 else "N/A (CPU)"
)
print(f"🖥️  Peak VRAM                    : {vram_display}")

>>> [1/7] Tải dữ liệu Spider...
Dataset URL: https://www.kaggle.com/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset
License(s): unknown
100% 96.0M/96.0M [00:00<00:00, 292MB/s]

Đang giải nén dữ liệu...
Đang tải công cụ chấm điểm chính thức của Spider...
Xong bước tải dữ liệu.
Thiết bị: cuda  |  Dùng FP16: True
>>> [2/7] Tiền xử lý dữ liệu...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Train: 8659 mẫu  |  Dev: 1034 mẫu
>>> [3/7] Đóng gói Dữ liệu (Tokenization)...
>>> [4/7] Khởi tạo CodeT5Small + Cấu hình LoRA Adapters...
trainable params: 6,684,672 || all params: 67,176,960 || trainable%: 9.9508

>>> [5/7] Bắt đầu huấn luyện với Adapter LoRA (5 epochs, LR=1e-03)...
Tổng optimizer steps: 1355 | Warmup: 81 steps


Epoch 01/5:   0%|          | 0/1083 [00:00<?, ?it/s]

📊 Epoch   1/5: Loss=2.5775 | LR=9.46e-04 | 197s
✅ Đã lưu checkpoint LoRA tại epoch 1


...

🗑️ Dọn dẹp checkpoint phụ thành công.

>>> [7/7] Inference & Spider Official Evaluation & Performance Metrics...


Inference:   0%|          | 0/65 [00:00<?, ?it/s]


>>> Spider Official Results (Trên Dev set):

>>> Đo Performance Metrics...


⏱ Đo Latency/Throughput:   0%|          | 0/65 [00:00<?, ?it/s]


📊 PERFORMANCE METRICS REPORT (LoRA VARIANT)
Chỉ số                               Giá trị
-------------------------------------------------------
Adapter Size (MB)                     56.46 MB
Latency (ms/sample)                  131.42 ms
Throughput (samples/s)                 7.61 s/s
Peak VRAM (MB)                    1667.32 MB


✅ Kết quả lưu thành công tại : /content/drive/MyDrive/CodeT5-small_LoRA
🏆 Epoch hoàn thành             : 5
💾 Adapter Weight Size          : 56.46 MB
⚡ Latency                      : 131.42 ms/sample
🚀 Throughput                   : 7.61 samples/s
🖥️  Peak VRAM                    : 1667.32 MB


In [ ]:
print("\n>>> [5/5] Kết quả đánh giá:")
!sed -i 's/conn = sqlite3.connect(db)/conn = sqlite3.connect(db)\n    conn.text_factory = lambda b: b.decode(errors="ignore")/' evaluation.py
!python evaluation.py --gold gold.txt --pred pred.txt --db spider_data/database --table spider_data/tables.json --etype all


>>> [5/5] Kết quả đánh giá:
medium pred: SELECT song_name ,  song_release_year FROM singer ORDER BY age DESC LIMIT 1
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

medium pred: SELECT song_name ,  song_release_year FROM singer ORDER BY age DESC LIMIT 1
medium gold: SELECT song_name ,  song_release_year FROM singer ORDER BY age LIMIT 1

medium pred: SELECT name ,  capacity FROM stadium ORDER BY AVG(Average) DESC LIMIT 1
medium gold: SELECT name ,  capacity FROM stadium ORDER BY average DESC LIMIT 1

medium pred: SELECT count(*) ,  stadium_id FROM concert GROUP BY stadium_id
medium gold: SELECT T2.name ,  count(*) FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id GROUP BY T1.stadium_id

eval_err_num:1
extra pred: SELECT name ,  capacity FROM stadium WHERE YEAR  =  2014 GROUP BY stadium_id ORDER BY count(*) DESC LIMIT 1
extra gold: SELECT T2.name ,  T2.capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadi